## RibonanzaNet Training Setup
This notebook section uses a medium_GNN-style data flow (sequence CSV plus labels CSV grouped by target_id), then creates a padded DataLoader and trains RibonanzaNet with masked token-wise regression.

In [24]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import random
import pickle
import yaml

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from pathlib import Path


class Config:
    DATA_PATH = Path("/scratch/phys/sin/rna-dataset")
    sample_sub = DATA_PATH / "sample_submission.csv"
    test_seq = DATA_PATH / "test_sequences.csv"
    train_labels = DATA_PATH / "cleaned" / "cleaned_labels.csv"
    train_sequences = DATA_PATH / "cleaned" / "cleaned_sequences.csv"
    validation_labels = DATA_PATH / "validation_labels.csv"
    validation_sequences = DATA_PATH / "validation_sequences.csv"
    model_config_path = "pairwise.yaml"
    pretrained_weights_path = "pytorch_model_fsdp.bin"
    
    max_len = 256
    batch_size = 1
    max_len_filter = 10000
    min_len_filter = 10
    seed = 94

config = Config()

# Set seed for reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(config.seed)

In [16]:
train_sequences = pd.read_csv(config.train_sequences)
train_labels = pd.read_csv(config.train_labels)
val_sequences = pd.read_csv(config.validation_sequences)
val_labels = pd.read_csv(config.validation_labels)

train_labels["pdb_id"] = train_labels.ID.str.rsplit('_', n=1, expand=True).iloc[:,0]
val_labels["pdb_id"] = val_labels.ID.str.rsplit('_', n=1, expand=True).iloc[:,0]

In [17]:
def making_data_dict(sequences_df: pd.DataFrame, labels_df: pd.DataFrame):
    grouped = labels_df.groupby("pdb_id")
    data = {}
    sequences, pdb_ids, all_xyz = [], [], []

    for pdb_id in tqdm(sequences_df["target_id"], desc="prepairing data"):
        if pdb_id not in grouped.groups:
            continue

        xyz = grouped.get_group(pdb_id)[["x_1", "y_1", "z_1"]].to_numpy(dtype="float32")

        sequence = list((sequences_df[sequences_df["target_id"]== pdb_id])["sequence"])[0]

        sequences.append(sequence)
        pdb_ids.append(pdb_id)
        all_xyz.append(xyz)
        
    data["pdb_ids"] = pdb_ids
    data["sequences"] = sequences
    data["all_xyz"] = all_xyz
    
    return data


#  Create Data Dictionaries

train_data_dict = making_data_dict(train_sequences, train_labels)
val_data_dict = making_data_dict(val_sequences, val_labels)

prepairing data: 100%|██████████| 28/28 [00:00<00:00, 439.94it/s]


In [18]:
class RNA3D_Dataset(Dataset):
    """
    A PyTorch Dataset for 3D RNA structures.
    """
    def __init__(self, data_dict, max_len=384):
        self.data = data_dict
        self.max_len = max_len
        self.nt_to_idx = {nt: i for i, nt in enumerate("ACGU")}

    def __len__(self):
        return len(self.data["sequences"])
    
    def __getitem__(self, idx):
        sequence = [self.nt_to_idx[nt] for nt in self.data["sequences"][idx]]
        sequence = torch.tensor(sequence, dtype=torch.long)
        xyz = torch.tensor(self.data["all_xyz"][idx], dtype=torch.float32)
        
        # If sequence is longer than max_len, randomly crop
        if len(sequence) > self.max_len:
            crop_start = np.random.randint(len(sequence) - self.max_len)
            crop_end = crop_start + self.max_len
            sequence = sequence[crop_start:crop_end]
            xyz = xyz[crop_start:crop_end]

        return {"sequence": sequence, "xyz": xyz}



# Create Dataset and DataLoaders

train_dataset = RNA3D_Dataset(train_data_dict, max_len=config.max_len)
val_dataset = RNA3D_Dataset(val_data_dict, max_len=config.max_len)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)



In [29]:
from Network import RibonanzaNet


class Config:
    def __init__(self, **entries):
        self.__dict__.update(entries)
        self.entries = entries

    def print(self):
        print(self.entries)


def load_config_from_yaml(file_path):
    with open(file_path, 'r') as file:
        cfg = yaml.safe_load(file)
    return Config(**cfg)



#  Model Definition
class FinetunedRibonanzaNet(RibonanzaNet):
    def __init__(self, config_obj, pretrained=False, dropout=0.1):
        config_obj.dropout = dropout
        super(FinetunedRibonanzaNet, self).__init__(config_obj)

        if pretrained:
            self.load_state_dict(
                torch.load(config.pretrained_weights_path, map_location="cpu")
            )

        self.dropout = nn.Dropout(p=0.0)
        self.xyz_predictor = nn.Linear(384, 3)

    def forward(self, src):
        sequence_features, _ = self.get_embeddings(
            src, torch.ones_like(src).long().to(src.device)
        )
        xyz_pred = self.xyz_predictor(sequence_features)
        return xyz_pred



# Initialize Model
model_cfg = load_config_from_yaml(config.model_config_path)
model = FinetunedRibonanzaNet(model_cfg, pretrained=True).cuda()

constructing 48 ConvTransformerEncoderLayers


In [33]:
def calculate_distance_matrix(X, Y, epsilon=1e-4):
    return ((X[:, None] - Y[None, :])**2 + epsilon).sum(dim=-1).sqrt()


def dRMSD(pred_x, pred_y, gt_x, gt_y, epsilon=1e-4, Z=10, d_clamp=None):
    pred_dm = calculate_distance_matrix(pred_x, pred_y)
    gt_dm = calculate_distance_matrix(gt_x, gt_y)

    mask = ~torch.isnan(gt_dm)
    mask[torch.eye(mask.shape[0], device=mask.device).bool()] = False

    diff_sq = (pred_dm[mask] - gt_dm[mask])**2 + epsilon
    if d_clamp is not None:
        diff_sq = diff_sq.clamp(max=d_clamp**2)

    return diff_sq.sqrt().mean() / Z


def local_dRMSD(pred_x, pred_y, gt_x, gt_y, epsilon=1e-4, Z=10, d_clamp=30):
    pred_dm = calculate_distance_matrix(pred_x, pred_y)
    gt_dm = calculate_distance_matrix(gt_x, gt_y)

    mask = (~torch.isnan(gt_dm)) & (gt_dm < d_clamp)
    mask[torch.eye(mask.shape[0], device=mask.device).bool()] = False

    diff_sq = (pred_dm[mask] - gt_dm[mask])**2 + epsilon
    return diff_sq.sqrt().mean() / Z


def dRMAE(pred_x, pred_y, gt_x, gt_y, epsilon=1e-4, Z=10):
    pred_dm = calculate_distance_matrix(pred_x, pred_y)
    gt_dm = calculate_distance_matrix(gt_x, gt_y)

    mask = ~torch.isnan(gt_dm)
    mask[torch.eye(mask.shape[0], device=mask.device).bool()] = False

    diff = torch.abs(pred_dm[mask] - gt_dm[mask])
    return diff.mean() / Z if Z != 0 else 0


def align_svd_mae(input_coords, target_coords, Z=10):
    mask = ~torch.isnan(target_coords.sum(dim=-1))
    input_coords = input_coords[mask]
    target_coords = target_coords[mask]

    centroid_input = input_coords.mean(dim=0, keepdim=True)
    centroid_target = target_coords.mean(dim=0, keepdim=True)

    input_centered = input_coords - centroid_input
    target_centered = target_coords - centroid_target

    cov_matrix = input_centered.T @ target_centered
    U, S, Vt = torch.svd(cov_matrix)
    R = Vt @ U.T

    if torch.det(R) < 0:
        Vt_adj = Vt.clone()
        Vt_adj[-1, :] = -Vt_adj[-1, :]
        R = Vt_adj @ U.T

    aligned_input = (input_centered @ R.T) + centroid_target
    return torch.abs(aligned_input - target_coords).mean() / Z

In [34]:
def train_model(model, train_dl, val_dl, epochs=50, cos_epoch=35, lr=3e-4, clip=1):
    optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.0, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=(epochs - cos_epoch) * len(train_dl),
    )

    best_val_loss = float("inf")
    best_preds = None

    for epoch in range(epochs):
        model.train()
        train_pbar = tqdm(train_dl, desc=f"Training Epoch {epoch+1}/{epochs}")
        running_loss = 0.0

        for idx, batch in enumerate(train_pbar):
            sequence = batch["sequence"].cuda()
            gt_xyz = batch["xyz"].squeeze().cuda()

            pred_xyz = model(sequence).squeeze()

            loss = dRMAE(pred_xyz, pred_xyz, gt_xyz, gt_xyz) + align_svd_mae(pred_xyz, gt_xyz)
            if torch.isnan(loss):
                print(f"Loss is nan on {epoch} epoch!")
                continue
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            optimizer.zero_grad()

            if (epoch + 1) > cos_epoch:
                scheduler.step()

            running_loss += loss.item()
            avg_loss = running_loss / (idx + 1)
            train_pbar.set_description(f"Epoch {epoch+1} | Loss: {avg_loss:.4f}")

        model.eval()
        val_loss = 0.0
        val_preds = []

        with torch.no_grad():
            for batch in val_dl:
                sequence = batch["sequence"].cuda()
                gt_xyz = batch["xyz"].squeeze().cuda()

                pred_xyz = model(sequence).squeeze()
                loss = dRMAE(pred_xyz, pred_xyz, gt_xyz, gt_xyz)
                val_loss += loss.item()

                val_preds.append((gt_xyz.cpu().numpy(), pred_xyz.cpu().numpy()))

        val_loss /= len(val_dl)
        print(f"Validation Loss (Epoch {epoch+1}): {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_preds = val_preds
            torch.save(model.state_dict(), config.save_weights_name)
            print(f"  -> New best model saved at epoch {epoch+1}")

    torch.save(model.state_dict(), config.save_weights_final)
    return best_val_loss, best_preds

In [ ]:
best_loss, best_predictions = train_model(
    model=model,
    train_dl=train_loader,
    val_dl=val_loader,
    epochs=20,
    cos_epoch=15,
    lr=3e-4,
    clip=1
)

Epoch 1 | Loss: 4.0767:   1%|          | 20/3124 [00:31<38:11,  1.35it/s]  

Loss is nan on 0 epoch!


Epoch 1 | Loss: 4.1430:   1%|          | 37/3124 [00:59<1:07:57,  1.32s/it]

Loss is nan on 0 epoch!


Epoch 1 | Loss: 4.1982:   3%|▎         | 81/3124 [02:21<1:16:30,  1.51s/it]

...

Epoch 1 | Loss: 4.0540:  12%|█▏        | 386/3124 [11:51<28:25,  1.61it/s]  

Loss is nan on 0 epoch!


Epoch 1 | Loss: 4.0799:  14%|█▍        | 430/3124 [13:15<49:45,  1.11s/it]  

Loss is nan on 0 epoch!


Epoch 1 | Loss: 4.1033:  16%|█▌        | 505/3124 [15:57<1:18:22,  1.80s/it]